# NIFTY50 Multi-Horizon LSTM Forecasting
**Objective:** Forecast NIFTY50 returns for 1–5 days ahead with prediction intervals and sectoral contagion analysis.

Set `HORIZON` below — everything else adapts automatically.

In [50]:
# ============================================================
# GLOBAL CONFIG — only change values here
# ============================================================
DATA_PATH  = "../nifty_final_dataset.csv"   # path to your CSV
SEQ_LEN    = 20                           # lookback window (days)
HORIZON    = 3                            # forecast horizon: 1–5
TOP_N      = 15                           # number of features to keep
EPOCHS     = 20
BATCH_SIZE = 32
LR         = 0.001
CONF_LIST  = [0.90, 0.95, 0.99]          # confidence levels for intervals

assert 1 <= HORIZON <= 5, "HORIZON must be between 1 and 5"

## 1. Imports

In [51]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression
from torch.utils.data import Dataset, DataLoader

In [52]:
import torch
import numpy as np
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

## 2. Data Preprocessing

In [53]:
# ── Load & encode categorical columns ──────────────────────
df = pd.read_csv(DATA_PATH)
df = pd.get_dummies(df, columns=["strongest_sector"])

# ── Feature selection via mutual information ───────────────
X_all = df.drop(columns=["target", "date"])
y_all = df["target"]

mi = mutual_info_regression(X_all, y_all, random_state=42)
mi_series = pd.Series(mi, index=X_all.columns).sort_values(ascending=False)
selected_features = mi_series.head(TOP_N).index.tolist()

print("Top features selected:")
print(mi_series.head(TOP_N).to_string())

# ── Scale features (fit on all data before splitting) ─────
# Note: scaler is fit here for saving; actual train/test split
# is done in Sec.3 strictly by time order (no leakage).
X = df[selected_features].values
y = y_all.values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Persist artifacts for inference ───────────────────────
pickle.dump(scaler, open("lstm_scaler.pkl", "wb"))
pickle.dump(selected_features, open("lstm_features.pkl", "wb"))
print("\nPreprocessing done ✅")

Top features selected:
low                          0.076528
open                         0.074600
vol_15                       0.057088
ret_15                       0.056922
close                        0.056500
high                         0.055061
ma_5                         0.053760
ma_20                        0.051602
dist_ma_5                    0.050567
corr_nifty_metal_ret_lag2    0.048905
it_ret_lag1                  0.048128
dist_ma_20                   0.044473
corr_nifty_log_ret           0.040303
auto_ret                     0.039091
bank_ret                     0.035866

Preprocessing done ✅


## 3. Sequence Creation (dynamic based on HORIZON)

In [54]:
def create_sequences(X, y, seq_len, horizon):
    """
    Build (input_window, target_window) pairs.
    X_seq shape : (n_samples, seq_len, n_features)
    y_seq shape : (n_samples, horizon)  — returns for t+1 … t+horizon
    """
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len - horizon + 1):
        X_seq.append(X[i : i + seq_len])
        y_seq.append(y[i + seq_len : i + seq_len + horizon])
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)

X_seq, y_seq = create_sequences(X_scaled, y, SEQ_LEN, HORIZON)

# ── Time-based split (no shuffling) ───────────────────────
total      = len(X_seq)
test_size  = int(0.10 * total)
calib_size = int(0.10 * total)
train_size = total - test_size - calib_size

X_train, y_train = X_seq[:train_size], y_seq[:train_size]
X_calib, y_calib = X_seq[train_size:train_size+calib_size], y_seq[train_size:train_size+calib_size]
X_test,  y_test  = X_seq[train_size+calib_size:], y_seq[train_size+calib_size:]

print(f"Sequences — Train: {len(X_train)} | Calib: {len(X_calib)} | Test: {len(X_test)}")
print(f"Each sample: input {X_train.shape[1:]} → output {y_train.shape[1:]}")

Sequences — Train: 1462 | Calib: 182 | Test: 182
Each sample: input (20, 15) → output (3,)


## 4. Model Definition (dynamic output = 2 × HORIZON)

In [55]:
class TSDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y)
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


class LSTMForecaster(nn.Module):
    """
    LSTM that predicts 2*horizon values:
      [raw_lower_1, raw_width_1, ..., raw_lower_H, raw_width_H]
    Bounds are derived as:
      lower_h = fc_out[2h]  ,  upper_h = lower_h + |fc_out[2h+1]|
    """
    def __init__(self, input_size, hidden=64, horizon=1):
        super().__init__()
        self.horizon = horizon
        self.lstm    = nn.LSTM(input_size, hidden, batch_first=True)
        self.fc      = nn.Linear(hidden, 2 * horizon)  # 2 outputs per day

    def forward(self, x):
        out, _ = self.lstm(x)
        out    = self.fc(out[:, -1, :])          # take last time-step

        lowers, uppers = [], []
        for h in range(self.horizon):
            l = out[:, 2*h]
            u = l + torch.abs(out[:, 2*h + 1])  # width always positive
            lowers.append(l)
            uppers.append(u)

        # Stack → (batch, horizon)
        return torch.stack(lowers, dim=1), torch.stack(uppers, dim=1)


def tube_loss(y_true, lower, upper, delta=0.01):
    """Penalises violations (y outside interval) + interval width."""
    coverage_penalty = torch.relu(lower - y_true) + torch.relu(y_true - upper)
    width_penalty    = delta * torch.abs(upper - lower)
    return torch.mean(coverage_penalty + width_penalty)


model = LSTMForecaster(input_size=len(selected_features), horizon=HORIZON)
print(model)
print(f"\nOutput dim: {2 * HORIZON}  (2 bounds × {HORIZON} days)")

LSTMForecaster(
  (lstm): LSTM(15, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=6, bias=True)
)

Output dim: 6  (2 bounds × 3 days)


## 5. Training Loop

In [56]:
train_loader = DataLoader(TSDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
optimizer    = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        lowers, uppers = model(X_batch)         # both: (batch, horizon)

        # Aggregate tube_loss across all horizon steps
        loss = sum(
            tube_loss(y_batch[:, h], lowers[:, h], uppers[:, h])
            for h in range(HORIZON)
        )
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch:02d}/{EPOCHS}  Loss: {epoch_loss:.4f}")

torch.save(model.state_dict(), "lstm_model.pth")
print("\nTraining complete ✅")

Epoch 01/20  Loss: 0.7165
Epoch 02/20  Loss: 0.2745
Epoch 03/20  Loss: 0.2240
Epoch 04/20  Loss: 0.2102
Epoch 05/20  Loss: 0.1823
Epoch 06/20  Loss: 0.1669
Epoch 07/20  Loss: 0.1561
Epoch 08/20  Loss: 0.1581
Epoch 09/20  Loss: 0.1536
Epoch 10/20  Loss: 0.1372
Epoch 11/20  Loss: 0.1376
Epoch 12/20  Loss: 0.1421
Epoch 13/20  Loss: 0.1378
Epoch 14/20  Loss: 0.1253
Epoch 15/20  Loss: 0.1231
Epoch 16/20  Loss: 0.1293
Epoch 17/20  Loss: 0.1218
Epoch 18/20  Loss: 0.1245
Epoch 19/20  Loss: 0.1272
Epoch 20/20  Loss: 0.1240

Training complete ✅


## 6. Conformal Calibration

In [57]:
# Compute non-conformity scores on calibration set.
# Score = max(lower - y, y - upper)  — positive means y fell outside interval.
model.eval()
calib_errors = []   # one error value per sample (use Day-1 as reference)

with torch.no_grad():
    X_c = torch.tensor(X_calib)
    lowers, uppers = model(X_c)           # (n_calib, horizon)

    for i in range(len(X_calib)):
        # Use Day-1 prediction for calibration scores
        l, u, y_true = lowers[i, 0].item(), uppers[i, 0].item(), y_calib[i, 0]
        calib_errors.append(max(l - y_true, y_true - u))

pickle.dump(calib_errors, open("lstm_errors.pkl", "wb"))
print(f"Calibration errors — mean: {np.mean(calib_errors):.4f}  "
      f"90th pct: {np.quantile(calib_errors, 0.90):.4f}")
print("Calibration done ✅")

Calibration errors — mean: -0.0105  90th pct: 0.0089
Calibration done ✅


## 7. Prediction Function

In [58]:
def predict(df_raw, model, scaler, features, calib_errors, horizon, conf_list):
    """
    Given the raw dataframe, produce conformal prediction intervals
    for each horizon day at each confidence level.

    Returns a dict: {conf → [(day, lower, upper, signal), ...]}
    """
    # Build the latest input window
    X_latest = scaler.transform(df_raw[features].values[-SEQ_LEN:])
    X_tensor = torch.tensor(X_latest, dtype=torch.float32).unsqueeze(0)  # (1, seq, feat)

    model.eval()
    with torch.no_grad():
        lowers, uppers = model(X_tensor)   # (1, horizon)
    lowers = lowers.squeeze(0).numpy()     # (horizon,)
    uppers = uppers.squeeze(0).numpy()

    results = {}
    for conf in conf_list:
        q_hat = np.quantile(calib_errors, conf)   # conformal adjustment
        day_results = []
        for h in range(horizon):
            l_adj = lowers[h] - q_hat
            u_adj = uppers[h] + q_hat
            mid   = (l_adj + u_adj) / 2
            signal = "BUY 📈" if mid > 0 else "SELL 📉"
            day_results.append((h + 1, l_adj, u_adj, signal))
        results[conf] = day_results

    return results


def sector_analysis(df_raw, top_n=3):
    """Return the top-N sectors by absolute last-day return."""
    sector_cols = ["bank_ret","it_ret","pharma_ret",
                   "auto_ret","fmcg_ret","metal_ret","energy_ret"]
    latest = df_raw.iloc[-1]
    sector_vals = {col: latest[col] for col in sector_cols if col in latest}
    return sorted(sector_vals.items(), key=lambda x: abs(x[1]), reverse=True)[:top_n]


print("Prediction functions defined ✅")

Prediction functions defined ✅


## 8. Output — Predictions & Sector Insights

In [ ]:
# ── Run predictions ────────────────────────────────────────
results = predict(df, model, scaler, selected_features,
                  calib_errors, HORIZON, CONF_LIST)

print(f"===== MULTI-DAY PREDICTION  (HORIZON = {HORIZON}) =====\n")
for conf, days in results.items():
    print(f"--- Confidence: {conf} ---")
    for day, l, u, signal in days:
        print(f"  Day {day} → [{l:.5f}, {u:.5f}]  {signal}")
    print()

# ── Sector contagion analysis ──────────────────────────────
print("===== SECTOR IMPACT (last trading day) =====")
top_sectors = sector_analysis(df)
for name, val in top_sectors:
    direction = "UP ↑" if val > 0 else "DOWN ↓"
    print(f"  {name:<20} {direction}  ({val:.4f})")

# ── Final narrative insight ────────────────────────────────
main_sector, main_val = top_sectors[0]
day1_signal = results[CONF_LIST[0]][0][3]   # Day-1 signal at lowest conf
cause = "upward" if main_val > 0 else "downward"

print(f"\n===== FINAL INSIGHT =====")
print(f"NIFTY outlook (Day 1): {day1_signal}")
print(f"Primary driver: {main_sector} exerting {cause} pressure ({main_val:.4f})")

===== MULTI-DAY PREDICTION  (HORIZON = 3) =====

--- Confidence: 0.9 ---
  Day 1 → [-0.09712, 0.05782]  SELL 📉
  Day 2 → [-0.07678, 0.06296]  SELL 📉
  Day 3 → [-0.05544, 0.13205]  BUY 📈

--- Confidence: 0.95 ---
  Day 1 → [-0.09996, 0.06066]  SELL 📉
  Day 2 → [-0.07962, 0.06580]  SELL 📉
  Day 3 → [-0.05828, 0.13489]  BUY 📈

--- Confidence: 0.99 ---
  Day 1 → [-0.10604, 0.06674]  SELL 📉
  Day 2 → [-0.08570, 0.07188]  SELL 📉
  Day 3 → [-0.06436, 0.14097]  BUY 📈

===== SECTOR IMPACT (last trading day) =====
  auto_ret             DOWN ↓  (-0.0282)
  bank_ret             DOWN ↓  (-0.0267)
  fmcg_ret             DOWN ↓  (-0.0181)

===== FINAL INSIGHT =====
NIFTY outlook (Day 1): SELL 📉
Primary driver: auto_ret exerting downward pressure (-0.0282)
